In [1]:
!pip install openpyxl

In [2]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [30]:
# ===========================
# Basic Libraries
# ===========================
import os
import re
import string
import random
import warnings
warnings.filterwarnings("ignore")

# ===========================
# Data Manipulation
# ===========================
import numpy as np
import pandas as pd

# ===========================
# Data Visualization
# ===========================
import matplotlib.pyplot as plt
import seaborn as sns

# ===========================
# Text Processing
# ===========================
import nltk

# Download only once
nltk.download('punkt')
nltk.download('stopwords')

# ===========================
# Scikit-Learn
# ===========================
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ===========================
# TensorFlow / Keras
# ===========================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\palla\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\palla\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [31]:
Oligofact_real_df = pd.read_excel(r"C:\Users\palla\Downloads\Real.xlsx")
Oligofact_fake_df = pd.read_excel(r"C:\Users\palla\Downloads\Fake.xlsx")

In [32]:
len(Oligofact_real_df)

5294

In [33]:
ISOT_real_df = pd.read_csv(r"C:\Users\palla\Downloads\True.csv")
ISOT_fake_df = pd.read_csv(r"C:\Users\palla\Downloads\Fake.csv")

In [34]:
ISOT_real_df["label"] = 0
ISOT_fake_df["label"] = 1


In [35]:
len(ISOT_real_df)

21417

In [36]:
ISOT_df  = pd.concat([ISOT_real_df, ISOT_fake_df], ignore_index=True)
ISOT_df = ISOT_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [37]:
ISOT_df.head(5)

,title,text,subject,date,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",1
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",1
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",1
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",0
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",1


In [38]:
ISOT_df["text"] = (ISOT_df["title"].fillna("")+" "+ISOT_df["text"].fillna(""))

In [39]:
ISOT_df = ISOT_df[
    ["text", "label"]
]

In [40]:
ISOT_df["language"] = "english"
ISOT_df["dataset"] = "ISOT"

In [41]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()         
        .str.lower()       
        .str.replace(" ", "_")
    )
    return df

Oligofact_real_df = clean_columns(Oligofact_real_df)
Oligofact_fake_df = clean_columns(Oligofact_fake_df)

In [42]:
Oligofact_real_df["label"] = 0
Oligofact_fake_df["label"] = 1

In [43]:
Oligofact_df  = pd.concat([Oligofact_real_df, Oligofact_fake_df], ignore_index=True)
Oligofact_df = Oligofact_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [44]:
Oligofact_df.head(10)

,gathering_date,news_date,url,domain,language,keywords,news_headline,news_original_text,english_translated_version,label
0,2023-05-30,2021-02-18,https://sputnik.by/video/20210218/1046947912/U...,sputnik.by,russian,"Ukrainian disintegration, Ukrainian statehood,...",«Утилизация людей»: Запад не позволит Грузии п...,«Первые дозы вакцины „Файзер“ (Pfizer/BioNTech...,“The first doses of the Fizer vaccine (Pfizer/...,1
1,2023-05-30,2018-03-06,https://ria.ru/world/20180306/1515866868.html,ria.ru,russian,"Sergei Skripal, Chemical weapons/attack, Consp...",Юлию Скрипаль могут убрать как ненужного свиде...,_x000D_\n Вся надежда на российских правоохран...,_x000D_ All hope for Russian law enforcement o...,1
2,2023-05-30,2021-04-14,https://de.rt.com/meinung/115826-mutmasslicher...,de.rt.com,hungarian,"Donbas, War in Ukraine, DNR",A NATO mukkanni sem tudott: Putyin Karabahban ...,https://t.stopnews.online/213963-nato-ne-uspel...,https://t.stopnews.online/213963-nato-ne-uspel...,1
3,2026-02-10 00:00:00,NaN,https://www.hs.fi/urheilu/art-2000011803252.html,www.hs.fi,finnish,"weather, fiemme, finland's, temperature",Kommentti|Iivo Niskasella ei ollut välineillää...,Val di Fiemme Suomen Iivo Niskanen paalutti lä...,Val di Fiemme Finland's Iivo Niskanen made it ...,0
4,2026-02-10 00:00:00,2026-02-07 00:00:00,https://www.aljazeera.net/ebusiness/2026/2/9/ب...,www.aljazeera.net,arabic,"international, syrian, democratic, government",ييدأ تأهيلهما خلال أسبوعين.. دمشق تستلم أكبر ح...,في خطوة جديدة لتطبيق بنود الاتفاق بين الحكومة ...,In a new step to implement the terms of the ag...,0
5,2026-02-10 00:00:00,NaN,https://archiv.hn.cz/c1-67842390-co-se-skryva-...,archiv.hn.cz,czech,"politics, czech, tomi, public",Co se skrv za plnem SPD na levnj potraviny? eb...,Tma cen potravin se pravideln vrac do esk veej...,The topic of food prices regularly returns to ...,0
6,2023-06-08 15:20:06.123000,NaN,https://www.bild.de/politik/leute/boris-johnso...,www.bild.de,German,"politics, chaos, british, prime",Boris Johnson: Aktuelle News und Videos,Schon wieder Chaos bei den britischen Konserva...,Chaos again for the British Conservatives! Bri...,0
7,2026-02-10 00:00:00,2026-02-02,https://www.europafm.ro/angajatii-din-peste-1-...,www.europafm.ro,romanian,"politics, warning, tuesday, public",Angajații din peste 1.300 de primării intră az...,"Grevă de avertisment marți, 10 februarie, în p...","Warning strike on Tuesday, February 10, in ove...",0
8,2026-02-10 00:00:00,2026-02-23,https://www.digi24.ro/stiri/actualitate/politi...,www.digi24.ro,romanian,"politics, secretary, general, prime",Claudiu Manda: Bolojan nu cântă de multe ori d...,"Secretarul general al PSD, Claudiu Manda, a af...","The Secretary General of the PSD, Claudiu Mand...",0
9,2026-02-10 00:00:00,NaN,https://archiv.hn.cz/c1-67843210-otazky-a-odpo...,archiv.hn.cz,czech,"politics, tuesday, november, internet",Otzky a odpovdi k omezen socilnch st dtem: Pro...,Na druh norov ter pipad mezinrodn den bezpenos...,"On the second Tuesday of November, the Interna...",0


In [45]:
print(Oligofact_df.columns.tolist())

['gathering_date', 'news_date', 'url', 'domain', 'language', 'keywords', 'news_headline', 'news_original_text', 'english_translated_version', 'label']


In [46]:
print(Oligofact_df[["news_headline", "news_original_text"]].isnull().sum())

news_headline         31
news_original_text    56
dtype: int64


In [47]:
Oligofact_df["text"] = (Oligofact_df["news_headline"].fillna("")+" "+Oligofact_df["news_original_text"].fillna(""))

In [48]:
Oligofact_df = Oligofact_df.dropna(subset=["news_headline", "news_original_text"], how="all")

In [49]:
Oligofact_df = Oligofact_df[
    ["text", "language", "label"]
]
Oligofact_df["dataset"] = "Oligofact"

In [50]:
len(Oligofact_df)

10190

In [51]:
final_df = pd.concat([Oligofact_df, ISOT_df],  ignore_index=True)
final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [26]:
final_df.head(10)

,text,language,label,dataset
0,White Man Terrorizes Black Shop Owner By Leav...,english,1,ISOT
1,Ground Zero Mosque Was NOT Defeated: Three Sto...,english,1,ISOT
2,Obama surprises Vice President Biden with Meda...,english,0,ISOT
3,Republicans Had Total Control Of This State A...,english,1,ISOT
4,EU to sign joint defense pact in show of post-...,english,0,ISOT
5,Germany's Schaeuble elected Bundestag speaker ...,english,0,ISOT
6,BREAKING: OBAMA POISED To Exact Revenge On PUT...,english,1,ISOT
7,RUSSIA Tells Sore Loser Obama To Produce Some ...,english,1,ISOT
8,Myanmar jails foreign journalists with Turkish...,english,0,ISOT
9,Why GOP Presidential Candidates Are Angry Ira...,english,1,ISOT


In [33]:
Oligofact_df["language"].value_counts()

language
russian        2634
czech          1283
romanian       1096
arabic          587
english         582
Spanish         480
hungarian       478
finnish         358
spanish         289
French          263
german          244
Slovak          215
Italian         213
Bulgarian       211
Armenian        202
German          200
italian         176
french          142
swedish         123
lithuanian      108
bulgarian        94
georgian         93
armenian         58
azerbaijani      39
slovak           22
Name: count, dtype: int64

In [27]:
len(final_df)

55088

In [28]:
final_df.to_csv(
    r"C:\Users\palla\Downloads\fakeNews_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

In [35]:
df = pd.read_csv(r"C:\Users\palla\Downloads\fakeNews_dataset.csv", encoding="utf-8-sig")
print(df.shape)

(55088, 4)
